# Pipeline — Master Orchestrator

Generates and submits a single SLURM job (`pipeline_run.py`) that runs the full
three-part H permeation pipeline end-to-end with no manual handoffs.

---

| Part | Script | Runs after |
|---|---|---|
| 1 — Surface NEB | `neb_run.py` | immediately |
| 3 — Bulk diffusivity | `diffusivity_run.py` | immediately (parallel with Part 1) |
| 2 — Permeation | `permeation_run.py` | after Part 1 **and** Part 3 complete |

**Data flow (automatic)**:
- Part 1 → `neb/ranked_barriers.json`, `neb/diss_vib_rates.json`, `adsorption/h_atom/`
- Part 3 → `results/diffusivity_arrhenius.json`, `results/lattice_params_vs_T.json`
- Part 2 reads all of the above at runtime — no `DH_DISS_EV`/`DH_ENTRY_EV` needed

**Workflow**:
1. Configure variables in Cell 2 (shared config for all three parts)
2. Run Cell 3 → generates `neb_run.py`
3. Run Cell 4 → generates `diffusivity_run.py`
4. Run Cell 5 → generates `permeation_run.py`
5. Run Cell 6 → generates `pipeline_run.py` + `pipeline_run.sh`, submits with `dry_run=True`
6. Set `dry_run=False` in Cell 6 to actually submit

**Individual notebooks** (`neb_calculation.ipynb`, `diffusivity.ipynb`, `permeation.ipynb`)
remain valid for partial re-runs, debugging, and post-run analysis.

## Cell 1 — Imports & sys.path

In [3]:
import os
import sys
import json

parent_dir = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from models.config import (
    MACE_MODEL_ASE,
    SLURM_DEFAULTS, BASE_DIR,
    N_REPLICAS, SPRING_CONST, NEB_FTOL,
)
from models.create_slurm import submit_slurm_job
from models.neb_workflow         import write_neb_run_script
from models.diffusivity_workflow import generate_diffusivity_scripts, generate_orchestrator_sh
from models.permeation_workflow  import generate_permeation_scripts, generate_permeation_sh
from models.pipeline_workflow    import generate_pipeline_scripts, generate_pipeline_sh

print('Imports OK.')

Imports OK.


## Cell 2 — Shared Configuration

Edit all-caps variables below before generating scripts.
These are the **single authoritative source** for all three parts.

> `DH_DISS_EV` and `DH_ENTRY_EV` are left as `None` — Part 2 auto-extracts them
> from `ranked_barriers.json` and `rate_dict_T{T}K.json` at runtime.

In [ ]:
# ── Root paths ────────────────────────────────────────────────────────────────
WORK_DIR = os.path.join(BASE_DIR, 'calculation')

# Script output paths
NEB_RUN_PY         = os.path.join(WORK_DIR, 'neb_run.py')
DIFFUSIVITY_RUN_PY = os.path.join(WORK_DIR, 'diffusivity_run.py')
PERMEATION_RUN_PY  = os.path.join(WORK_DIR, 'permeation_run.py')
PIPELINE_RUN_PY    = os.path.join(WORK_DIR, 'pipeline_run.py')
PIPELINE_RUN_SH    = os.path.join(WORK_DIR, 'pipeline_run.sh')

# ── Temperatures shared by all parts ─────────────────────────────────────────
TEMPERATURES = [500, 600, 700, 800, 900]   # K

# ═══════════════════════════════════════════════════════════════════════════════
# Part 1 — Surface NEB (neb_run.py)
# ═══════════════════════════════════════════════════════════════════════════════
BULK_MIN_PATH = os.path.join(BASE_DIR, 'structures/bulk_min.lammps')
SLAB_DIR      = os.path.join(WORK_DIR, 'slabs')
ADS_DIR       = os.path.join(WORK_DIR, 'adsorption')
NEB_DIR       = os.path.join(WORK_DIR, 'neb')

E_H2_GAS = -6.790499 # eV — set to H2 single-point energy
MILLER         = (1, 1, 1)
LAYERS         = 12
VACUUM         = 15.0   # Å
LAT_REPEAT     = (5, 6)
SEP_MIN        = 2.5    # Å
SEP_MAX        = 6.0    # Å
GRAPH_DIST_MIN = 2
PROX_CUTOFF    = 5.0    # Å
N_IMAGES       = N_REPLICAS
SPRING_K       = SPRING_CONST
NEB_FTOL_VAL   = NEB_FTOL
H_HEIGHT       = 1.5    # Å

NEB_GPU_SLURM = dict(SLURM_DEFAULTS, partition='multigpu', time='04:00:00')
NEB_CPU_SLURM = dict(SLURM_DEFAULTS, partition='short',
                     gpu=None, cpus_per_task=16, time='12:00:00')
NEB_VIB_SLURM = dict(SLURM_DEFAULTS, partition='short',
                     gpu=None, cpus_per_task=8,  time='06:00:00')

# ═══════════════════════════════════════════════════════════════════════════════
# Part 3 — Bulk diffusivity (diffusivity_run.py)
# ═══════════════════════════════════════════════════════════════════════════════
INPUT_STRUCTURES = [
    os.path.join(BASE_DIR, 'structures/bulk_seed.lammps'),
]
N_H_VALUES    = [1, 3, 5, 7, 10]
DIFF_TEMPS    = TEMPERATURES    # share temperature grid with Parts 1 & 2

NVT_WALL_TIME = '24:00:00'
NVT_CUTOFF    = '23:55:00'
GPU_PARTITION = 'multigpu'
GPU_TIME      = NVT_WALL_TIME

TIMESTEP_PS   = 0.0005
TAU_T_PS      = 0.1
N_EQUIL_STEPS = 2_000_000
N_PROD_STEPS  = 5_000_000
THERMO_EVERY  = 1000
DUMP_EVERY    = 1000
VELOCITY_SEED = 42
RESTART_EVERY = 100_000

# ═══════════════════════════════════════════════════════════════════════════════
# Part 2 — Permeation (permeation_run.py)
# ═══════════════════════════════════════════════════════════════════════════════
RELAXED_SLAB_PATH  = os.path.join(WORK_DIR, 'slabs', 'slab_relaxed.lammps')
SURFACE_SITES_JSON = os.path.join(WORK_DIR, 'slabs', 'surface_sites.json')
PHASE2_H_DIR       = os.path.join(WORK_DIR, 'adsorption', 'h_atom')
SUB_NEB_DIR        = os.path.join(WORK_DIR, 'neb_subsurface')
VIB_DIR            = os.path.join(WORK_DIR, 'vibrations')
RESULTS_DIR        = os.path.join(WORK_DIR, 'results')

P_VALS_PA     = [1e3, 1e4, 1e5, 5e5, 1e6]   # Pa
A0_M          = 3.52e-10   # m — fallback lattice parameter
L_M           = 1e-3       # m — membrane thickness
NX, NY        = 40, 40
SEED          = 42
KMC_MAX_STEPS = 500_000

# D0/ED: auto-loaded from diffusivity_arrhenius.json at runtime (Part 3 output)
_diff_json = os.path.join(RESULTS_DIR, 'diffusivity_arrhenius.json')
if os.path.exists(_diff_json):
    with open(_diff_json) as _f: _diff = json.load(_f)
    D0_M2S = _diff['D0_m2s']; E_D_EV = _diff['E_D_eV']
    print(f'Pre-loaded D0={D0_M2S:.3e} m²/s  E_D={E_D_EV:.3f} eV from {_diff_json}')
else:
    D0_M2S = 1.5e-7; E_D_EV = 0.40   # placeholders

# DH values: auto-extracted at runtime by permeation_run.py
DH_DISS_EV  = None
DH_ENTRY_EV = None

PERM_GPU_SLURM = dict(SLURM_DEFAULTS, partition='multigpu', time='04:00:00')
PERM_NEB_SLURM = dict(SLURM_DEFAULTS, partition='short',
                      gpu=None, cpus_per_task=16, time='12:00:00')
PERM_VIB_SLURM = dict(SLURM_DEFAULTS, partition='short',
                      gpu=None, cpus_per_task=8,  time='06:00:00')

# ── Pipeline orchestrator SLURM ───────────────────────────────────────────────
PIPE_JOB_NAME      = 'pipeline_orch'
PIPE_PARTITION     = 'west'
PIPE_CPUS_PER_TASK = 4
PIPE_MEM           = '16G'
PIPE_TIME          = '30-00:00:00'   # west partition 30-day maximum
PIPE_OPENMPI_VER   = SLURM_DEFAULTS.get('openmpi_ver', '')
PIPE_CUDA_VER      = SLURM_DEFAULTS.get('cuda_version', '')
PIPE_CONDA_ENV     = SLURM_DEFAULTS.get('conda_env', 'mace_env')
PIPE_LD_PATHS      = SLURM_DEFAULTS.get('ld_paths', [])

print('Config loaded.')
print(f'  WORK_DIR     : {WORK_DIR}')
print(f'  TEMPERATURES : {TEMPERATURES}')
print(f'  N_H_VALUES   : {N_H_VALUES}')
print(f'  E_H2_GAS     : {E_H2_GAS}')
print(f'  D0_M2S       : {D0_M2S:.3e} m²/s   E_D_EV: {E_D_EV:.3f} eV')

## Cell 3 — Generate `neb_run.py` (Part 1)

In [9]:
write_neb_run_script(
    bulk_min_path  = BULK_MIN_PATH,
    work_dir       = WORK_DIR,
    e_h2_gas       = E_H2_GAS,
    slab_dir       = SLAB_DIR,
    ads_dir        = ADS_DIR,
    neb_dir        = NEB_DIR,
    miller         = MILLER,
    layers         = LAYERS,
    vacuum         = VACUUM,
    lat_repeat     = LAT_REPEAT,
    sep_min        = SEP_MIN,
    sep_max        = SEP_MAX,
    graph_dist_min = GRAPH_DIST_MIN,
    prox_cutoff    = PROX_CUTOFF,
    n_images       = N_IMAGES,
    spring_const   = SPRING_K,
    neb_ftol       = NEB_FTOL_VAL,
    h_height       = H_HEIGHT,
    gpu_slurm_cfg  = NEB_GPU_SLURM,
    neb_slurm_cfg  = NEB_CPU_SLURM,
    vib_slurm_cfg  = NEB_VIB_SLURM,
    out_py         = NEB_RUN_PY,
)
print(f'Written: {NEB_RUN_PY}')

OSError: [Errno 30] Read-only file system: '/projects'

## Cell 4 — Generate `diffusivity_run.py` (Part 3)

In [6]:
generate_diffusivity_scripts(
    input_structures = INPUT_STRUCTURES,
    n_h_values       = N_H_VALUES,
    temperatures     = DIFF_TEMPS,
    work_dir         = WORK_DIR,
    nvt_wall_time    = NVT_WALL_TIME,
    cutoff           = NVT_CUTOFF,
    gpu_partition    = GPU_PARTITION,
    gpu_time         = GPU_TIME,
    timestep_ps      = TIMESTEP_PS,
    tau_t_ps         = TAU_T_PS,
    n_equil_steps    = N_EQUIL_STEPS,
    n_prod_steps     = N_PROD_STEPS,
    thermo_every     = THERMO_EVERY,
    dump_every       = DUMP_EVERY,
    velocity_seed    = VELOCITY_SEED,
    restart_every    = RESTART_EVERY,
    out_py           = DIFFUSIVITY_RUN_PY,
)
print(f'Written: {DIFFUSIVITY_RUN_PY}')

OSError: [Errno 30] Read-only file system: '/projects'

## Cell 5 — Generate `permeation_run.py` (Part 2)

In [7]:
generate_permeation_scripts(
    work_dir           = WORK_DIR,
    relaxed_slab_path  = RELAXED_SLAB_PATH,
    surface_sites_json = SURFACE_SITES_JSON,
    phase2_h_dir       = PHASE2_H_DIR,
    sub_neb_dir        = SUB_NEB_DIR,
    vib_dir            = VIB_DIR,
    results_dir        = RESULTS_DIR,
    temperatures       = TEMPERATURES,
    p_vals_pa          = P_VALS_PA,
    a0_m               = A0_M,
    l_m                = L_M,
    d0_m2s             = D0_M2S,
    e_d_ev             = E_D_EV,
    dh_diss_ev         = DH_DISS_EV,
    dh_entry_ev        = DH_ENTRY_EV,
    nx                 = NX,
    ny                 = NY,
    seed               = SEED,
    kmc_max_steps      = KMC_MAX_STEPS,
    gpu_slurm_cfg      = PERM_GPU_SLURM,
    neb_slurm_cfg      = PERM_NEB_SLURM,
    vib_slurm_cfg      = PERM_VIB_SLURM,
    n_images           = N_IMAGES,
    spring_const       = SPRING_K,
    neb_ftol           = NEB_FTOL_VAL,
    out_py             = PERMEATION_RUN_PY,
)
print(f'Written: {PERMEATION_RUN_PY}')

FileNotFoundError: [Errno 2] No such file or directory: '/projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel/calculation/permeation_run.py'

## Cell 6 — Generate `pipeline_run.py` + `pipeline_run.sh` and submit

This is the entry point: one `sbatch pipeline_run.sh` starts the entire pipeline.

Set `dry_run=False` when ready to submit.

In [8]:
generate_pipeline_scripts(
    neb_run_py         = NEB_RUN_PY,
    diffusivity_run_py = DIFFUSIVITY_RUN_PY,
    permeation_run_py  = PERMEATION_RUN_PY,
    work_dir           = WORK_DIR,
    out_py             = PIPELINE_RUN_PY,
)

generate_pipeline_sh(
    orch_job_name      = PIPE_JOB_NAME,
    orch_partition     = PIPE_PARTITION,
    orch_cpus_per_task = PIPE_CPUS_PER_TASK,
    orch_mem           = PIPE_MEM,
    orch_time          = PIPE_TIME,
    orch_openmpi_ver   = PIPE_OPENMPI_VER,
    orch_cuda_version  = PIPE_CUDA_VER,
    orch_conda_env     = PIPE_CONDA_ENV,
    orch_ld_paths      = PIPE_LD_PATHS,
    out_py             = PIPELINE_RUN_PY,
    out_sh             = PIPELINE_RUN_SH,
)

# Preview pipeline_run.py
print('\n--- pipeline_run.py (first 30 lines) ---')
with open(PIPELINE_RUN_PY) as _f:
    for _i, _l in enumerate(_f):
        if _i >= 30: break
        print(_l, end='')

# Submit (set dry_run=False to actually submit)
job_id = submit_slurm_job(PIPELINE_RUN_SH, dry_run=True)
print(f'\nPipeline job ID: {job_id}')

OSError: [Errno 30] Read-only file system: '/projects'